In [4]:
import pandas as pd
import os

import re


import h5py
import json

from tqdm import tqdm
import numpy as np

## prepare mapping between splits and species names

In [6]:
dir = '/mnt/nfs_dna/chepurova/'
metadata_df = pd.read_csv(os.path.join(dir, 'mammals_data/mammals_dataset_metadata3.txt'), sep='\t')
metadata_df

,assembly_accession,organism_name,n_autosomes,number_of_x,number_of_y
0,GCF_004115215.2,Ornithorhynchus anatinus,21,5,20
1,GCF_015852505.1,Tachyglossus aculeatus,31,5,9
2,GCF_902635505.1,Sarcophilus harrisii,6,1,1
3,GCF_011100635.1,Trichosurus vulpecula,9,10,0
4,GCF_030445035.1,Dasypus novemcinctus,46,1,0
...,...,...,...,...,...
120,GCF_002776525.5,Piliocolobus tephrosceles,21,1,1
121,GCF_903992535.2,Arvicola amphibius,17,1,0
122,GCF_019320065.1,Cervus canadensis,33,1,1
123,GCF_028023285.1,Balaenoptera ricei,54,1,0


In [7]:
TEST_SPECIES = ["Sarcophilus harrisii", "Monodelphis domestica", "Manis pentadactyla", "Equus asinus", "Sus scrofa", "Camelus dromedarius", 
        "Phyllostomus discolor", "Myotis daubentonii", "Suncus etruscus", "Lepus europaeus", "Ochotona princeps", 
        "Sciurus carolinensis", "Nycticebus coucang", "Lemur catta", "Cynocephalus volans", 'Elephas maximus indicus', 'Loxodonta africana',
        "Choloepus didactylus", "Ornithorhynchus anatinus", "Tachyglossus aculeatus"]

VALID_SPECIES = ["Lynx canadensis", "Neofelis nebulosa", "Eubalaena glacialis", "Balaenoptera musculus", "Cervus canadensis", "Dama dama",
                "Meriones unguiculatus", "Chionomys nivalis", "Jaculus jaculus", "Callithrix jacchus"]

no_autosomes = ["Mastomys coucha", 'Ursus arctos']

metadata_df = metadata_df[(metadata_df['number_of_x'] != 0) & (metadata_df['number_of_y'] != 0) & (metadata_df['n_autosomes'] != 0)]

TRAIN_SPECIES = metadata_df[metadata_df['organism_name'].apply(lambda x: (x not in VALID_SPECIES) and (x not in TEST_SPECIES))].organism_name.to_list()

In [15]:
split2organism_name = {
    "train": TRAIN_SPECIES,
    "valid": VALID_SPECIES,
    "test": TEST_SPECIES
}

with open(os.path.join(dir, 'mammals_data/split2organism_name.json'), 'w') as f:
    json.dump(split2organism_name, f)

In [16]:
print(len(TRAIN_SPECIES))
print(len(VALID_SPECIES))
print(len(TEST_SPECIES))

28
10
20


## check if all the species have haploid chromosome sets 

In [17]:
def read_chunk_from_hdf5(hdf5_path, dataset_name, start, length):
    with h5py.File(hdf5_path, 'r') as hdf5_file:
        dset = hdf5_file[dataset_name]
        chunk = dset[start:start + length].astype(str)
        return ''.join(chunk)

In [21]:
chr_df = pd.read_csv(os.path.join(dir, 'mammals_data/mammals_diploid_chr_number.tsv'))

In [22]:
for split in ['train', 'valid', 'test']:
    for species in split2organism_name[split]:
        print(species)
        assembly_id = metadata_df[metadata_df['organism_name'] == species]['assembly_accession'].values[0]
        
        with h5py.File(os.path.join(dir, f"mammals_data/{split}.h5"), 'r') as hdf5_file:
            autosomes = []
            sex_chromosomes = []
            
            for chr in list(hdf5_file[assembly_id].keys()):
               chr_name = re.findall(r'chromosome\s+([0-9]{1,2}|[A-Za-z]{1,2})', chr)[0]
               if chr_name != "Y" and chr_name != "X":
                   try:
                       autosomes.append(int(chr_name))
                   except:
                       autosomes.append(chr_name)
               else:
                   sex_chromosomes.append(chr_name)

            autosomes = list(set(autosomes))
            sex_chromosomes = list(set(sex_chromosomes))
            autosomes.sort()
            
            print("Sex chromosomes in dataset:", " ".join(sex_chromosomes))
            print("Autosomes in dataset:", autosomes)
            print("Diploid number of chromosomes in dataset:", len(autosomes) * 2 + len(sex_chromosomes))
            print("Diploid number of chromosomes:", chr_df[chr_df['species_name'] == species]['diploid_chr_number'].values[0])
            print(chr_df[chr_df['species_name'] == species]['diploid_chr_number'].values[0] == len(autosomes) * 2 + len(sex_chromosomes))
            print()

Macaca mulatta
Sex chromosomes in dataset: X Y
Autosomes in dataset: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Diploid number of chromosomes in dataset: 42
Diploid number of chromosomes: 42
True

Papio anubis
Sex chromosomes in dataset: X Y
Autosomes in dataset: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Diploid number of chromosomes in dataset: 42
Diploid number of chromosomes: 42
True

Pan paniscus
Sex chromosomes in dataset: X Y
Autosomes in dataset: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
Diploid number of chromosomes in dataset: 48
Diploid number of chromosomes: 48
True

Pongo abelii
Sex chromosomes in dataset: X Y
Autosomes in dataset: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
Diploid number of chromosomes in dataset: 48
Diploid number of chromosomes: 48
True

Homo sapiens
Sex chromosomes in dataset: X Y
Autosomes in dataset: [1, 2, 3, 4

In [49]:
# metadata_df[metadata_df['organism_name'] == 'Ursus arctos']

In [50]:
# metadata_df[metadata_df['organism_name'] == 'Mastomys coucha']

## Merging mammals, human and mouse data

In [24]:
def create_links(source_file, target_file, source_filename, path="/"):
    """
    Create external links from all groups and datasets in source_file to target_file,
    placing them directly in the root of the target file (no prefixes).
    """
    for key in source_file:
        item = source_file[key]
        target_path = f"{path}{key}"
        if isinstance(item, h5py.Group):
            # Create group in the target and recursevely add links of its subgroups
            if key not in target_file:
                grp = target_file.create_group(key)
            else:
                grp = target_file[key]
            create_links(item, grp, source_filename, path=target_path + "/")
        else:
            target_file[key] = h5py.ExternalLink(source_filename, target_path)


for split in ['train', 'valid', 'test']:
    merged_file = os.path.join(dir, f'merged_data/merged_links_{split}.h5')
    with h5py.File(merged_file, 'w') as merged_file:

        mammals_file = os.path.join(dir, f'mammals_data/{split}.h5')
        with h5py.File(mammals_file, 'r') as f:
            create_links(f, merged_file, source_filename=mammals_file)

        human_file = os.path.join(dir, f'human_data/{split}.h5')
        with h5py.File(human_file, 'r') as f:
            create_links(f, merged_file, source_filename=human_file)

        mouse_file = os.path.join(dir, f'mouse_data/{split}.h5')
        with h5py.File(mouse_file, 'r') as f:
            create_links(f, merged_file, source_filename=mouse_file)

    print(f"Merged file 'mammals_gender_data/merged_links_{split}.h5' created.")

Merged file 'mammals_gender_data/merged_links_train.h5' created.
Merged file 'mammals_gender_data/merged_links_valid.h5' created.
Merged file 'mammals_gender_data/merged_links_test.h5' created.


In [27]:
with h5py.File(os.path.join(dir, 'merged_data/merged_links_train.h5'), 'r') as f:
    print(len(f.keys()))
    print("GCF_902655055.1" in f.keys())
    print(f['GCF_902655055.1'].keys())

81
True
<KeysViewHDF5 ['NC_062278.1 chromosome 1 ', 'NC_062279.1 chromosome 2 ', 'NC_062280.1 chromosome 3 ', 'NC_062281.1 chromosome 4 ', 'NC_062282.1 chromosome 5 ', 'NC_062283.1 chromosome 6 ', 'NC_062284.1 chromosome 7 ', 'NC_062285.1 chromosome 8 ', 'NC_062286.1 chromosome 9 ', 'NC_062287.1 chromosome 10 ', 'NC_062288.1 chromosome 11 ', 'NC_062289.1 chromosome 12 ', 'NC_062290.1 chromosome 13 ', 'NC_062291.1 chromosome 14 ', 'NC_062292.1 chromosome 15 ', 'NC_062293.1 chromosome 16 ', 'NC_062294.1 chromosome 17 ', 'NC_062295.1 chromosome 18 ', 'NC_062296.1 chromosome X ', 'NC_062297.1 chromosome Y ']>


## Merging metadata

In [29]:
for split in ['train', 'valid', 'test']:
    
    human_metadata_df = pd.read_csv(os.path.join(dir, f'human_data/{split}.csv'), index_col=0)
    mouse_metadata_df = pd.read_csv(os.path.join(dir, f'mouse_data/{split}.csv'), index_col=0)

    species = split2organism_name[split]
    mammals_metadata_df = metadata_df[metadata_df['organism_name'].isin(species)]
    mammals_metadata_df = mammals_metadata_df[['organism_name', 'assembly_accession']]
    mammals_metadata_df.rename(columns={'assembly_accession': 'assembly_accession/sample_id'}, inplace=True)
    mammals_metadata_df['sex'] = None
    mammals_metadata_df['diploid'] = False

    human_metadata_df['organism_name'] = 'Homo sapiens'
    human_metadata_df.rename(columns={'sample': 'assembly_accession/sample_id'}, inplace=True)
    human_metadata_df['diploid'] = True
    human_metadata_df['sex'] = human_metadata_df['sex'].map({1: '0', 2: '1'})

    mouse_metadata_df['organism_name'] = 'Mus musculus'
    mouse_metadata_df.rename(columns={'strain_name': 'assembly_accession/sample_id'}, inplace=True)
    mouse_metadata_df.rename(columns={'gender': 'sex'}, inplace=True)
    mouse_metadata_df['sex'] = mouse_metadata_df['sex'].map({'M': 0, 'F': 1})
    mouse_metadata_df['diploid'] = False
    mouse_metadata_df = mouse_metadata_df[['organism_name', 'assembly_accession/sample_id', 'sex', 'diploid']]
    
    all_metadata_df = pd.concat([mammals_metadata_df, human_metadata_df, mouse_metadata_df])
    all_metadata_df.reset_index(drop=True, inplace=True)

    all_metadata_df.to_csv(os.path.join(dir, f'merged_data/{split}_merged_metadata.csv'), index=False)

In [30]:
all_metadata_df

,organism_name,assembly_accession/sample_id,sex,diploid
0,Ornithorhynchus anatinus,GCF_004115215.2,None,False
1,Tachyglossus aculeatus,GCF_015852505.1,None,False
2,Sarcophilus harrisii,GCF_902635505.1,None,False
3,Lemur catta,GCF_020740605.2,None,False
4,Nycticebus coucang,GCF_027406575.1,None,False
5,Loxodonta africana,GCF_030014295.1,None,False
6,Equus asinus,GCF_016077325.2,None,False
7,Sus scrofa,GCF_000003025.6,None,False
8,Camelus dromedarius,GCF_036321535.1,None,False
9,Ochotona princeps,GCF_030435755.1,None,False


In [31]:
with open(os.path.join(dir, 'mammals_data/split2organism_name.json'), 'r') as f:
    mammals_split2organism_name = json.load(f)

mammals_split2organism_name['valid'].append('Homo sapiens')
mammals_split2organism_name['test'].append('Homo sapiens')
mammals_split2organism_name['valid'].append('Mus musculus')
mammals_split2organism_name['test'].append('Mus musculus')

with open(os.path.join(dir, 'merged_data/split2organism_name.json'), 'w') as f:
    json.dump(mammals_split2organism_name, f)

In [32]:
print(len(mammals_split2organism_name['valid']))
print(len(mammals_split2organism_name['test']))
print(len(mammals_split2organism_name['train']))


12
22
28


In [36]:
for split in ['train', 'valid', 'test']:
    metadata_df = pd.read_csv(os.path.join(dir, f'merged_data/{split}_merged_metadata.csv'))
    metadata_assembly_ids = metadata_df['assembly_accession/sample_id'].unique()

    with h5py.File(os.path.join(dir, f'merged_data/merged_links_{split}.h5'), 'r') as f:
        assembly_ids = list(f.keys())

        assert all([assembly_id in assembly_ids for assembly_id in metadata_assembly_ids])
        print(split, len(metadata_assembly_ids), len(metadata_df['organism_name'].unique()))

train 79 28
valid 30 12
test 54 22


## Checking chromosomes names

In [37]:
for split in ['train', 'valid', 'test']:
    with h5py.File(os.path.join(dir, f'mammals_data/merged_links_{split}.h5'), 'r') as f:
        for assembly_id in f.keys():
            for chr in list(f[assembly_id].keys()):
                if "Y" in chr:
                   assert ('chromosome Y' in chr) or ('chrY' in chr)

                if "X" in chr:
                   assert ('chromosome X' in chr) or ('chrX' in chr)
            #    chr_name = re.findall(r'chromosome\s+([0-9]{1,2}|[A-Za-z]{1,2})', chr)[0]


## Checking dataset class

In [41]:
# from mammals_gender_dataset import MultiSpeciesGenderDataChunkedDataset

# for split in ['train']:
#     dataset = MultiSpeciesGenderDataChunkedDataset(data_path=os.path.join(dir, 'merged_data/'), split_name=split, max_n_samples=100, chunk_size=10, n_chunks=10, force_sampling_from_y=True)
#     for elem in dataset:
#         print(elem)
    